# 03 · Анализ

Числа и таблицы по результатам прогона — без DOLFINx, только файлы: работает и на
ноутбуке с копией папки `runs/<имя>/`.

1. Какие прогоны есть и что в них посчитано (манифест `run.json`).
2. Временные ряды (`series.csv`) и механические биомаркеры.
3. Карты активации (`activation.npz`): время активации, скорость проведения, APD по регионам,
   блок проведения, реституция.
4. Снимки полей: статистика по регионам.
5. Серия: сводная таблица «параметры → биомаркеры».
6. Экспорт таблиц в `runs/<имя>/analysis/`.

Графики — в `04_visualization.ipynb`.

In [1]:
# Общая настройка: пакет cardiac_em и помощники ноутбуков доступны из любой папки проекта
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "notebooks" / "nbtools.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break
from nbtools import ROOT, RUNS, run_stream, style  # noqa: E402

print("корень проекта:", ROOT)

корень проекта: /home/user020/cardio2d


In [2]:
import json
import numpy as np
import pandas as pd
from cardiac_em.analysis import biomarkers as bm
from cardiac_em.analysis import open_run, open_sweep
from nbtools import list_runs

pd.set_option("display.precision", 4)
pd.DataFrame(list_runs())

,папка,тип,статус,модель,"t, мс","счёт, с"
0,a,прогон,finished,rogers_mcculloch,1500.0,397.014
1,nb_quick,прогон,finished,tnnpm,450.0,234.485
2,nb_quick_sweep,серия,0/6 готово,tnnpm,NaN,NaN
3,night,серия,3/3 готово,tnnpm,NaN,NaN
4,night_1hz,серия,3/3 готово,tnnpm,NaN,NaN
5,night_smoke,прогон,finished,tnnpm,20.0,21.440
6,sweep_tmax,серия,0/6 готово,rogers_mcculloch,NaN,NaN
7,tnnpm_ischemia,прогон,finished,tnnpm,400.0,307.371


## 1. Прогон

Имя папки в `runs/` (или полный путь к папке с `run.json`).

In [3]:
RUN_DIR = RUNS / "night_1hz" / "run_000"          # ночная серия: RUNS / "night" / "run_000"

run = open_run(RUN_DIR)
m = run.manifest
print(f"статус: {run.status},  счёт {m.get('elapsed_s')} с,  "
      f"t = {m['schedule']['t_start_ms']:g} → {m['schedule']['t_end_ms']:g} мс")
print(f"модель клетки: {m['models']['cell']},  материал: {m['models']['material']}")
print(f"перенос T_act: {m['models']['transfer']}")
for part in ("electric", "mechanical"):
    g = run.meshes[part]
    print(f"сетка {part:10}: {g['nx']}×{g['ny']} на {g['lx_mm']}×{g['ly_mm']} мм, {g['n_cells']} ячеек")
env = m.get("environment", {})
print(f"окружение: DOLFINx {env.get('dolfinx')}, рангов MPI {env.get('mpi_size')}, "
      f"git {env.get('git', {}).get('commit', '—')[:10]}")
for w in run.warnings:
    print("  [!]", w)

статус: finished,  счёт 18667.713 с,  t = 0 → 10000 мс
модель клетки: tnnpm,  материал: transversely_isotropic_exponential
перенос T_act: осреднение по ячейке (точный)
сетка electric  : 160×160 на 8.0×8.0 мм, 25600 ячеек
сетка mechanical: 40×40 на 8.0×8.0 мм, 1600 ячеек
окружение: DOLFINx 0.10.0, рангов MPI 8, git 2d80704a1e


### Параметры прогона

In [4]:
cfg = run.config
tb = cfg["tissue_base"]
print("T_MAX =", tb["active"]["t_max"], "кПа;  D =", tb["conduction"]["d_long"], "/",
      tb["conduction"]["d_trans"], "мм²/мс;  волокна", tb["conduction"]["fiber_angle_deg"], "°")
print("параметры клетки для всей ткани:", cfg.get("cell_params") or "по умолчанию")
print("стимулы, мс:", cfg["stimulus"]["times_ms"])
regions = pd.DataFrame([{"№": i + 1, "имя": r.get("name"), "форма": r["shape"],
                         **r.get("params", {}), **r.get("overrides", {})}
                        for i, r in enumerate(cfg.get("regions", []))])
regions if not regions.empty else print("регионов нет — однородная ткань")

T_MAX = 60.0 кПа;  D = 0.15 / 0.05 мм²/мс;  волокна 0.0 °
параметры клетки для всей ткани: по умолчанию
стимулы, мс: [1.0, 1001.0, 2001.0, 3001.0, 4001.0, 5001.0, 6001.0, 7001.0, 8001.0, 9001.0]


,№,имя,форма,cx,cy,r,cell:K_o,cell:ATP_i,cell:KmATP
0,1,ишемия,circle,5.0,4.0,2.0,9.4,4.0,0.38


## 2. Временные ряды и механика

`series.csv` — по строке на механический шаг. `t_act_mech_*` — сила, переданная клетками
(у TNNPM — при нулевой скорости волокна), `t_act_actual_*` — действующее напряжение с учётом
сила–скорость.

In [5]:
series = run.series(as_frame=True)
series.describe().T[["min", "max", "mean"]]

,min,max,mean
t_ms,0.0000,10000.0000,5000.0000
mech_index,1.0000,10000.0000,5000.5000
u_min,-85.9260,10.6291,-66.8382
u_max,-71.8341,56.0744,-53.3618
t_act_electric_max,0.0005,67.7613,16.9880
t_act_electric_integral,0.0179,3912.8952,825.0423
t_act_mech_max,0.0005,67.7430,16.9382
t_act_mech_integral,0.0179,3912.8952,825.0423
t_act_actual_max,0.0005,67.6844,15.8559
t_act_actual_integral,0.0178,3911.3302,823.0013


In [6]:
mech = bm.mechanics_summary(run.series())
pd.Series(mech, name="механика").to_frame()

,механика
peak_t_act_kpa,6.7743e+01
time_to_peak_t_act_ms,2.4500e+02
peak_sigma_xx_kpa,7.8143e+01
time_to_peak_sigma_ms,2.6200e+02
sigma_rise_kpa,6.4746e+01
min_lambda_f,1.2053e+00
max_shortening,4.1547e-02
t_act_time_integral_kpa_mm2_ms,8.2512e+06


## 3. Карты активации

По каждому узлу электрической сетки и каждому удару: время активации (пересечение порога
модели снизу вверх), реполяризации (APD на уровне `apd_level`, по умолчанию 90 %) и пик.

In [7]:
try:
    maps = run.activation()
except FileNotFoundError as exc:
    maps = None
    print(exc)
else:
    print(f"ударов: {maps.n_beats},  узлов: {len(maps.coords)},  порог {maps.threshold:g},  "
          f"APD{int(round(maps.apd_level * 100))}")

ударов: 10,  узлов: 25921,  порог -40,  APD90


In [8]:
def beat_table(maps, window=None):
    rows = []
    for k in range(maps.n_beats):
        act, apd = maps.act[:, k], maps.apd[:, k]
        s = bm.activation_summary(act)
        a = bm.apd_summary(apd)
        rows.append({"удар": k + 1, **s,
                     "CV_x, мм/мс": bm.conduction_velocity(maps.coords, act, axis=0, window=window),
                     "APD mean, мс": a["mean_ms"], "APD min": a["min_ms"], "APD max": a["max_ms"],
                     "дисперсия APD": a["dispersion_ms"]})
    return pd.DataFrame(rows)

CV_WINDOW = None      # (x_min, x_max) мм — окно для скорости; None — средние 40 % области
beats = beat_table(maps, CV_WINDOW) if maps is not None else pd.DataFrame()
beats

,удар,first_ms,last_ms,total_activation_time_ms,fraction_activated,"CV_x, мм/мс","APD mean, мс",APD min,APD max,дисперсия APD
0,1,1.6937,14.0925,12.3988,1.0,0.6561,245.0413,223.9581,256.0091,32.0510
1,2,1001.6936,1015.6094,13.9158,1.0,0.5799,244.4831,221.6366,255.2524,33.6157
2,3,2001.6938,2015.5908,13.8970,1.0,0.5809,244.5822,221.8093,255.2244,33.4151
3,4,3001.6939,3015.5746,13.8807,1.0,0.5818,244.6874,222.0007,255.2361,33.2354
4,5,4001.6939,4015.5609,13.8670,1.0,0.5827,244.7771,222.1956,255.2517,33.0561
5,6,5001.6940,5015.5487,13.8548,1.0,0.5836,244.8535,222.3912,255.2656,32.8744
6,7,6001.6940,6015.5346,13.8406,1.0,0.5845,244.9224,222.5833,255.2786,32.6952
7,8,7001.6940,7015.5219,13.8279,1.0,0.5853,244.9799,222.7689,255.2878,32.5190
8,9,8001.6940,8015.5103,13.8163,1.0,0.5862,245.0421,222.9632,255.3023,32.3391
9,10,9001.6940,9015.4997,13.8056,1.0,0.5870,245.1026,223.1674,255.3168,32.1494


Локальная скорость проведения |∇t_act|⁻¹ (медиана по области — устойчивее, чем среднее:
в зоне стимула градиент почти нулевой) и APD по регионам (0 — базовая ткань, *i* — регион *i*
конфигурации).

In [9]:
if maps is not None and maps.n_beats:
    g = run.meshes["electric"]
    hx, hy = g["lx_mm"] / g["nx"], g["ly_mm"] / g["ny"]
    BEAT = -1                         # удар для CV и APD: 0 — первый, -1 — последний
    b = BEAT % maps.n_beats
    cv = bm.cv_field(maps.grid(maps.act[:, b]), hx, hy)
    print(f"локальная CV (удар {b + 1}): медиана {np.nanmedian(cv):.3f} мм/мс, "
          f"5–95 %: {np.nanpercentile(cv, 5):.3f}–{np.nanpercentile(cv, 95):.3f}")
    names = {0: "базовая ткань", **{i + 1: r.get("name") or f"регион {i + 1}"
                                    for i, r in enumerate(cfg.get("regions", []))}}
    by_region = pd.DataFrame(bm.apd_summary(maps.apd[:, b], maps.region)["by_region"]).T
    by_region.index = [names.get(i, i) for i in by_region.index]
    display(by_region.rename(columns={"mean": "APD mean, мс", "std": "std", "min": "min", "max": "max", "n": "узлов"}))

локальная CV (удар 10): медиана 0.745 мм/мс, 5–95 %: 0.352–1.466


,узлов,"APD mean, мс",std,min,max
базовая ткань,20900.0,248.5279,5.4975,226.2670,255.3168
ишемия,5021.0,230.8450,4.8281,223.1674,241.4021


In [10]:
if maps is not None and maps.n_beats > 1:
    for k in range(maps.n_beats - 1):
        blk = bm.conduction_block(maps.act, k)
        print(f"удар {k + 1} → {k + 2}: не проведено в {blk['n_blocked']} узлах "
              f"({100 * blk['fraction_blocked']:.1f} %)")
    rest = bm.restitution(maps.act, maps.repol)
    print(f"пар (DI, APD) для реституции: {len(rest['di_ms'])}")
elif maps is not None:
    print("один удар — блок проведения и реституция требуют нескольких")

удар 1 → 2: не проведено в 0 узлах (0.0 %)
удар 2 → 3: не проведено в 0 узлах (0.0 %)
удар 3 → 4: не проведено в 0 узлах (0.0 %)
удар 4 → 5: не проведено в 0 узлах (0.0 %)
удар 5 → 6: не проведено в 0 узлах (0.0 %)
удар 6 → 7: не проведено в 0 узлах (0.0 %)
удар 7 → 8: не проведено в 0 узлах (0.0 %)
удар 8 → 9: не проведено в 0 узлах (0.0 %)
удар 9 → 10: не проведено в 0 узлах (0.0 %)
пар (DI, APD) для реституции: 233289


## 4. Снимки: статистика по регионам

Для каждого снимка — средние по регионам: потенциал на электрической сетке, растяжение
волокна и действующее напряжение на механической.

In [11]:
V_NAME = "V" if run.manifest["models"]["cell"].startswith("tnnpm") else run.manifest["models"]["cell_states"][0]
rows = []
for snap in run.snapshots():
    for r in np.unique(snap["m_region"]):
        me, mm = snap["e_region"] == r, snap["m_region"] == r
        rows.append({"t, мс": snap.t_ms, "регион": int(r),
                     f"{V_NAME} средн.": snap.state(V_NAME)[me].mean(),
                     "λ_f средн.": snap["m_lambda_f"][mm].mean(),
                     "λ_f мин.": snap["m_lambda_f"][mm].min(),
                     "T_act средн., кПа": snap["m_t_act"][mm].mean()})
snap_table = pd.DataFrame(rows)
snap_table if not snap_table.empty else print("снимков нет (output.snapshot_times_ms)")

,"t, мс",регион,V средн.,λ_f средн.,λ_f мин.,"T_act средн., кПа"
0,5.0,0,-38.7004,1.2575,1.2575,0.0002
1,5.0,1,-72.4884,1.2575,1.2575,0.0005
2,10.0,0,-6.9364,1.2575,1.2575,0.0002
3,10.0,1,-14.6894,1.2575,1.2575,0.0005
4,20.0,0,10.6036,1.2575,1.2575,0.0007
5,20.0,1,10.1550,1.2575,1.2575,0.0006
6,50.0,0,12.2398,1.2575,1.2568,0.2249
7,50.0,1,8.9918,1.2577,1.2572,0.1828
8,100.0,0,11.7135,1.2570,1.2478,7.4701
9,100.0,1,8.0078,1.2595,1.2533,7.1890


## 5. Серия

Сводная таблица по серии: по строке на точку — её параметры и биомаркеры. Функция `metrics`
определяет, какие числа нужны; её можно менять как угодно.

In [12]:
SWEEP_DIR = RUNS / "night_1hz"  # ночная серия: RUNS / "night"
SWEEP_BEAT = -1                      # удар для метрик серии: -1 — последний (установившийся)

def metrics(r):
    out = bm.mechanics_summary(r.series())
    try:
        mp = r.activation()
        b = SWEEP_BEAT % mp.n_beats
        out.update(bm.activation_summary(mp.act[:, b]))
        a = bm.apd_summary(mp.apd[:, b], mp.region)
        out["APD mean, мс"] = a["mean_ms"]
        for reg, st in a["by_region"].items():
            out[f"APD регион {reg}"] = st["mean"]
    except FileNotFoundError:
        pass
    return out

if (SWEEP_DIR / "sweep.json").exists():
    sweep = open_sweep(SWEEP_DIR)
    sweep_table = pd.DataFrame(sweep.table(metrics))
    done = sum(p["status"] == "finished" for p in sweep.points)
    print(f"посчитано точек: {done} из {len(sweep.points)}")
    if sweep_table.empty:
        sweep_table = None
        print("запустите серию в 01_run.ipynb (раздел 6, RUN_SWEEP или RUN_SWEEP_MPI)")
    else:
        display(sweep_table)
else:
    sweep_table = None
    print(f"серии в {SWEEP_DIR.relative_to(ROOT)} нет — запустите её в 01_run.ipynb (раздел 6)")

посчитано точек: 3 из 3


,name,regions.0.overrides.cell:K_o,regions.0.overrides.cell:ATP_i,regions.0.overrides.cell:KmATP,peak_t_act_kpa,time_to_peak_t_act_ms,peak_sigma_xx_kpa,time_to_peak_sigma_ms,sigma_rise_kpa,min_lambda_f,max_shortening,t_act_time_integral_kpa_mm2_ms,first_ms,last_ms,total_activation_time_ms,fraction_activated,"APD mean, мс",APD регион 0,APD регион 1
0,run_000,9.4,4.0,0.38,67.7430,245.0,78.1427,262.0,64.7464,1.2053,0.0415,8.2512e+06,9001.6940,9015.4997,13.8056,1.0,245.1026,248.5279,230.8450
1,run_001,8.0,4.5,0.35,67.8005,245.0,78.3399,264.0,64.9436,1.2196,0.0301,8.5556e+06,9001.6975,9013.3628,11.6653,1.0,249.5517,251.7861,240.2510
2,run_002,5.4,6.8,0.09,67.9216,246.0,78.6605,266.0,65.2643,1.2206,0.0293,9.3876e+06,9001.7019,9011.5864,9.8845,1.0,258.5803,258.5714,258.6173


## 6. Экспорт

Таблицы — в `runs/<имя>/analysis/` (CSV открываются в Excel/Origin), сводка — в JSON.

In [13]:
out_dir = run.dir / "analysis"
out_dir.mkdir(exist_ok=True)
series.to_csv(out_dir / "series.csv", index=False)
if not beats.empty:
    beats.to_csv(out_dir / "beats.csv", index=False)
if not snap_table.empty:
    snap_table.to_csv(out_dir / "snapshots_by_region.csv", index=False)
if sweep_table is not None:
    sweep_table.to_csv(SWEEP_DIR / "sweep_table.csv", index=False)

summary = {"run": str(run.dir.relative_to(ROOT)), "mechanics": mech,
           "beats": beats.to_dict(orient="records")}
(out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=float),
                                      encoding="utf-8")
print("записано в", out_dir.relative_to(ROOT), ":", sorted(p.name for p in out_dir.iterdir()))

записано в runs/night_1hz/run_000/analysis : ['beats.csv', 'series.csv', 'snapshots_by_region.csv', 'summary.json']
